In [1]:
#TTS stoftware installs
%pip install "pip<24.1"
%pip install "torch==2.5.1" "torchaudio==2.5.1"
%pip install git+https://github.com/liyaodev/fairseq.git
%pip install faiss-cpu ffmpeg-python loguru praat-parselmouth pyworld torchcrepe edge-tts nest_asyncio av librosa scipy soundfile
%pip install rvc-python --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 63.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 43.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 39.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.4 

In [2]:
#Tailscale client software install
#installing libraries
%pip install websockets 
%pip install PySocks

#installing tailscale
!curl -fsSL https://tailscale.com/install.sh | sh

Installing Tailscale for ubuntu jammy, using method apt
+ mkdir -p --mode=0755 /usr/share/keyrings
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.noarmor.gpg
+ tee /usr/share/keyrings/tailscale-archive-keyring.gpg
+ chmod 0644 /usr/share/keyrings/tailscale-archive-keyring.gpg
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.tailscale-keyring.list
+ tee /etc/apt/sources.list.d/tailscale.list
# Tailscale packages for ubuntu jammy
deb [signed-by=/usr/share/keyrings/tailscale-archive-keyring.gpg] https://pkgs.tailscale.com/stable/ubuntu jammy main
+ chmod 0644 /etc/apt/sources.list.d/tailscale.list
+ apt-get update
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://pkgs.tailscale.com/stable/ubuntu jammy InRelease                 
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://cli.github.com/packages stable/ma

In [5]:
#importing tts model
from google.colab import drive
import os
from rvc_python.infer import RVCInference

drive.mount('/content/drive')
MODEL_PATH = "/content/drive/MyDrive/mochigo/anya.pth"

print("\nBooting up PyTorch on NVIDIA GPU...")
rvc = RVCInference(device="cuda:0")
rvc.load_model(MODEL_PATH)
rvc.set_params(
    f0up_key=10,       
    f0method="rmvpe", 
    index_rate=0,
    filter_radius=3,
    resample_sr=0,
    rms_mix_rate=0.25,
    protect=0.33
)
print("The mochi has awaken!")

Mounted at /content/drive

Booting up PyTorch on NVIDIA GPU...
is_half:True, device:cuda:0
File hubert_base.pt not found, start loading...
File hubert_base.pt successfully loaded.
File rmvpe.pt not found, start loading...
File rmvpe.pt successfully loaded.
File rmvpe.onnx not found, start loading...
File rmvpe.onnx successfully loaded.


/usr/local/lib/python3.12/dist-packages/rvc_python/modules/vc/modules.py:103: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.cpt = torch.load(sid, map_location="cpu")
/u

gin_channels: 256 self.spk_embed_dim: 109
Model anya.pth loaded.
The mochi has awaken!


In [15]:
import asyncio
import edge_tts
from rvc_python.infer import RVCInference
import os
import re
import soundfile as sf
import numpy as np
from google.colab import drive
import nest_asyncio
from IPython.display import Audio, display, clear_output


nest_asyncio.apply()

drive.mount('/content/drive')

MODEL_PATH = "/content/drive/MyDrive/mochigo/anya.pth"
INDEX_PATH = "/content/drive/MyDrive/mochigo/anya.index"

print("\nBooting up System on NVIDIA GPU. Please hold...")

rvc = RVCInference(device="cuda:0")
rvc.load_model(MODEL_PATH)
rvc.set_params(
    f0up_key=10,
    f0method="rmvpe",   
    index_rate=0,       
    filter_radius=3,
    resample_sr=0,
    rms_mix_rate=0.25,
    protect=0.33
)

async def warmup():
    print("Pre-loading features into VRAM...")
    await edge_tts.Communicate("a", "en-US-AriaNeural").save("warm.mp3")
    rvc.infer_file("warm.mp3", "warm.wav")

asyncio.run(warmup())
# -------------------------------

def split_by_language(text):
    jp_pattern = r'([\u3000-\u303f\u3040-\u309f\u30a0-\u30ff\uff00-\uff9f\u4e00-\u9faf\u3400-\u4dbf]+)'
    raw_chunks = re.split(jp_pattern, text)
    parsed_chunks = []
    for chunk in raw_chunks:
        if not chunk.strip(): continue
        parsed_chunks.append(('ja' if re.search(jp_pattern, chunk) else 'en', chunk.strip()))
    return parsed_chunks

def cleanup():
    for f in os.listdir('/content'):
        if f.startswith("temp_") or f.startswith("warm") or f == "final_mochi.wav":
            try: os.remove(f)
            except: pass



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Booting up System on NVIDIA GPU. Please hold...


/usr/local/lib/python3.12/dist-packages/rvc_python/modules/vc/modules.py:103: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.cpt = torch.load(sid, map_location="cpu")
/u

gin_channels: 256 self.spk_embed_dim: 109
Model anya.pth loaded.
Pre-loading features into VRAM...


/usr/local/lib/python3.12/dist-packages/fairseq/checkpoint_utils.py:340: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(f, map_location=torch.device("cpu")

In [30]:
#making connection to fastapi server over tailscape 

import os
import getpass

AUTH_KEY = getpass.getpass("Enter your Tailscale Auth Key: ")
#AUTH_KEY = " "

!pkill tailscaled

!nohup tailscaled --tun=userspace-networking --socks5-server=localhost:1055 > /dev/null 2>&1 &
!sleep 2

os.system(f'tailscale up --authkey="{AUTH_KEY}"')

!tailscale status

# 1. Map your Tailscale IP directly to your MagicDNS hostname
!echo "100.127.64.87 note-d1.tail8b0d7e.ts.net" >> /etc/hosts

# 2. Verify it was added
!tail -n 1 /etc/hosts


import subprocess
import socks
import socket
from websockets.asyncio.client import connect

socks.set_default_proxy(socks.SOCKS5, "localhost", 1055, True)
socket.socket = socks.socksocket
print("Proxy patched successfully.")

100.71.254.40    06baf004fc76  FunnyKoalaBear@  linux    offline                     
100.123.134.1    00ce3356a50b  FunnyKoalaBear@  linux    offline, last seen 17d ago  
100.91.246.41    4526abc9b118  FunnyKoalaBear@  linux    offline, last seen 20d ago  
100.126.89.93    46ca69f1fc8c  FunnyKoalaBear@  linux    offline, last seen 17d ago  
100.92.9.17      652e5b934f40  FunnyKoalaBear@  linux    offline, last seen 20d ago  
100.86.112.123   6f681557de82  FunnyKoalaBear@  linux    offline, last seen 20d ago  
100.122.142.114  9ef1e30dfec7  FunnyKoalaBear@  linux    offline, last seen 18d ago  
100.122.232.35   f3e588f12dec  FunnyKoalaBear@  linux    offline, last seen 16d ago  
100.100.86.4     iphone183     FunnyKoalaBear@  iOS      offline, last seen 20d ago  
100.127.64.87    note-d1       FunnyKoalaBear@  windows  -                           
100.104.54.9     raspberrypi   FunnyKoalaBear@  linux    -                           
100.127.64.87 note-d1.tail8b0d7e.ts.net
Proxy patched 

In [31]:
!tailscale up
!tailscale status 

100.71.254.40    06baf004fc76  FunnyKoalaBear@  linux    -                           
100.123.134.1    00ce3356a50b  FunnyKoalaBear@  linux    offline, last seen 17d ago  
100.91.246.41    4526abc9b118  FunnyKoalaBear@  linux    offline, last seen 20d ago  
100.126.89.93    46ca69f1fc8c  FunnyKoalaBear@  linux    offline, last seen 17d ago  
100.92.9.17      652e5b934f40  FunnyKoalaBear@  linux    offline, last seen 20d ago  
100.86.112.123   6f681557de82  FunnyKoalaBear@  linux    offline, last seen 20d ago  
100.122.142.114  9ef1e30dfec7  FunnyKoalaBear@  linux    offline, last seen 18d ago  
100.122.232.35   f3e588f12dec  FunnyKoalaBear@  linux    offline, last seen 16d ago  
100.100.86.4     iphone183     FunnyKoalaBear@  iOS      offline, last seen 20d ago  
100.127.64.87    note-d1       FunnyKoalaBear@  windows  -                           
100.104.54.9     raspberrypi   FunnyKoalaBear@  linux    -                           


In [29]:
!tailscale down

error fetching current status: Failed to connect to local Tailscale daemon for /localapi/v0/status; not running? Error: dial unix /var/run/tailscale/tailscaled.sock: connect: no such file or directory


In [11]:
#networking class
class WSClient:

    def __init__(self, url: str):
        self.url = url
        self.websocket = None

    async def connect(self):
        self.websocket = await connect(self.url)
        print("Connected!")

    async def send(self, text: str):
        await self.websocket.send(text)
    
    async def recv(self):
        return await self.websocket.recv()
    
    async def sendAudio(self, audio_bytes): 
        #need to convert numpy array into bytes first 
        await self.websocket.send(audio_bytes)
        print("Audio sent!")

    async def sendWav(self, file):
        
        #compress wav
        subprocess.run(["ffmpeg", "-y", "-i", "final_mochi.wav", "final_mochi.mp3"], capture_output=True)
        
        #open wav
        with open("final_mochi.mp3", "rb") as f:
            wav_bytes = f.read()

        #compress to mp3 

        await self.websocket.send(wav_bytes)
        print("WAV file sent")

    
class Audio():
    def __init__(self):
        self.audioFile = "audio.mp4"
        self.text = "hi how are you doing today"

    def wake(self):
        #continuous function that checks if user is talking 
        
        #simulating it by waiting for input 
        start = input("Press enter to start talking")
        print("Wake triggered!")


    def record(self):
        #calls mic.py to record from rsp-script
        #sends recorded file to main.py 
        self.text = input("Enter your query: ")
        return self.text


audio = Audio()
wsclient = WSClient("ws://note-d1.tail8b0d7e.ts.net:8000/ws/tts")

In [ ]:

#main program
if __name__ == "__main__":
    cleanup()
    clear_output() # Clears the messy boot logs so you start with a clean screen!
    print("READY. Mochi is listening...\n")
    print("="*40)
    
    # --- THE INTERACTIVE LOOP ---
    while True:
        user_input = input("\nWhat should Mochi say? (or type 'exit'): ")
        
        if user_input.lower() == 'exit':
            print("\nShutting down AI...")
            break
            
        if not user_input.strip():
            continue
            
        print("Processing on GPU...")
        chunks = split_by_language(user_input)
        combined_audio = []
        target_sr = None
        
        try:
            for i, (lang, content) in enumerate(chunks):
                base_audio = f"temp_base_{i}.mp3"
                final_audio = f"temp_anya_{i}.wav"
                
                # 1. Route to Native Speakers
                if lang == 'en':
                    asyncio.run(edge_tts.Communicate(content, "en-US-AriaNeural").save(base_audio))
                else:
                    asyncio.run(edge_tts.Communicate(content, "ja-JP-NanamiNeural").save(base_audio))
                    
                # 2. Apply Anime Filter
                rvc.infer_file(base_audio, final_audio)
                
                # 3. Read the audio data
                data, sr = sf.read(final_audio)
                combined_audio.append(data)
                target_sr = sr
            
            if combined_audio:
                # 4. Stitch chunks into one file
                final_mix = np.concatenate(combined_audio)
                sf.write("final_mochi.wav", final_mix, target_sr)
                
                # 5. Clear the screen and pop up the new audio player
                clear_output(wait=True)
                print("="*40)
                print(f"Mochi: {user_input}")
                display(Audio("final_mochi.wav", autoplay=True))
                
        except Exception as e:
            print(f"\n[ERROR] {e}")
            
            cleanup()

In [ ]:
# function for tts
async def tts(llmOutput): # Make sure this is async if using await inside Colab!
    chunks = split_by_language(llmOutput)
    combined_audio = []
    target_sr = None
    
    # Print the chunks so we can see exactly what the splitter is doing!
    print(f"DEBUG Chunks: {chunks}") 
    
    for i, (lang, content) in enumerate(chunks):
        content = content.strip()
        if not content:
            continue
            
        base_audio = f"temp_base_{i}.mp3"
        final_audio = f"temp_anya_{i}.wav"
        
        # --- THE FIX: Wrap the TTS generation in its own try/except ---
        try:
            # 1. Route to Native Speakers
            if lang == 'en':
                await edge_tts.Communicate(content, "en-US-AriaNeural").save(base_audio)
            else:
                await edge_tts.Communicate(content, "ja-JP-NanamiNeural").save(base_audio)
        except Exception as e:
            # If a chunk fails (like a stray parenthesis), skip it and keep going!
            print(f"[WARNING] Skipping unpronounceable chunk '{content}': {e}")
            continue 
        # --------------------------------------------------------------
            
        # 2. Apply Anime Filter
        rvc.infer_file(base_audio, final_audio)
        
        # 3. Read the audio data
        data, sr = sf.read(final_audio)
        combined_audio.append(data)
        target_sr = sr
        
    try:
        if combined_audio:
            # 4. Stitch chunks into one file
            final_mix = np.concatenate(combined_audio)
            sf.write("final_mochi.wav", final_mix, target_sr)
            
            # 5. Clear the screen and pop up the new audio player
            clear_output(wait=True)
            print("="*40)
            print(f"Mochi: {llmOutput}")
            display(Audio("final_mochi.wav", autoplay=True))
            
    except Exception as e:
        print(f"\n[CRITICAL ERROR] Failed to stitch audio: {e}")
        cleanup()



In [61]:
import asyncio

async def run_tts():

    await wsclient.connect()

    while 1:
        #recieve llm output
        llmOut = await wsclient.recv()
        print(llmOut)

        #process llm output with tts 
        await tts(llmOut)

        #local wav file playback 
        #display(Audio("final_mochi.wav", autoplay=True))
        
        #send tts output to server 
        await wsclient.sendWav("final_mochi.wav")

        #loopback 
        await asyncio.sleep(0.01)


In [62]:
#making connection to fastapi server over tailscape 

import os
import getpass

AUTH_KEY = getpass.getpass("Enter your Tailscale Auth Key: ")
#AUTH_KEY = " "

!pkill tailscaled

!nohup tailscaled --tun=userspace-networking --socks5-server=localhost:1055 > /dev/null 2>&1 &
!sleep 2

os.system(f'tailscale up --authkey="{AUTH_KEY}"')

!tailscale status

# 1. Map your Tailscale IP directly to your MagicDNS hostname
!echo "100.127.64.87 note-d1.tail8b0d7e.ts.net" >> /etc/hosts

# 2. Verify it was added
!tail -n 1 /etc/hosts


import subprocess
import socks
import socket
from websockets.asyncio.client import connect

socks.set_default_proxy(socks.SOCKS5, "localhost", 1055, True)
socket.socket = socks.socksocket
print("Proxy patched successfully.")

100.71.254.40    06baf004fc76  FunnyKoalaBear@  linux    offline                     
100.123.134.1    00ce3356a50b  FunnyKoalaBear@  linux    offline, last seen 17d ago  
100.91.246.41    4526abc9b118  FunnyKoalaBear@  linux    offline, last seen 20d ago  
100.126.89.93    46ca69f1fc8c  FunnyKoalaBear@  linux    offline, last seen 17d ago  
100.92.9.17      652e5b934f40  FunnyKoalaBear@  linux    offline, last seen 20d ago  
100.86.112.123   6f681557de82  FunnyKoalaBear@  linux    offline, last seen 20d ago  
100.122.142.114  9ef1e30dfec7  FunnyKoalaBear@  linux    offline, last seen 18d ago  
100.122.232.35   f3e588f12dec  FunnyKoalaBear@  linux    offline, last seen 17d ago  
100.100.86.4     iphone183     FunnyKoalaBear@  iOS      offline, last seen 21d ago  
100.127.64.87    note-d1       FunnyKoalaBear@  windows  -                           
100.104.54.9     raspberrypi   FunnyKoalaBear@  linux    -                           
100.127.64.87 note-d1.tail8b0d7e.ts.net
Proxy patched 

In [ ]:
try:
    #await run_mochigo()
    await run_tts()
except KeyboardInterrupt:
    print("\nShutting down MochiGo...")

    #closing tailscape
    #subprocess.run("tailscale down")

Mochi: Oh! You remembered our counting lesson! "One, two, three" is 一、二、三! And then you asked my name! My name is MochiGo! "Name" is 名前 (なまえ). What is your name? Can you tell me in English?


WAV file sent
